# 01 — MediaPipe Face Mesh Landmarks

Notebook này minh họa cách phát hiện **468 face landmarks** từ ảnh/video bằng **MediaPipe Face Mesh**.

## Nội dung
1. Cài đặt và import thư viện
2. Phát hiện landmarks từ ảnh tĩnh
3. Visualize landmarks lên ảnh
4. Xem cấu trúc dữ liệu JSON output
5. Phát hiện landmarks từ video (tuỳ chọn)

## 1. Import thư viện

In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc vào PYTHONPATH
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.preprocess.landmark_mediapipe import (
    landmarks_to_dict,
    draw_landmarks_on_image,
    process_image,
)
from src.utils.io import ensure_dir, save_json, load_json
from src.utils.logger import get_logger

logger = get_logger('notebook_01')
print('✅ Import thành công!')

## 2. Thiết lập đường dẫn

In [ ]:
# Đặt đường dẫn ảnh đầu vào
# Thay bằng ảnh thực của bạn nếu có
INPUT_IMAGE = Path('data/samples/test.jpg')
OUTPUT_DIR = Path('outputs/landmarks2d')

ensure_dir(OUTPUT_DIR)

if not INPUT_IMAGE.exists():
    # Tạo ảnh mẫu giả để demo nếu chưa có ảnh thật
    print(f'⚠️  Ảnh {INPUT_IMAGE} chưa tồn tại.')
    print('Vui lòng đặt ảnh khuôn mặt vào data/samples/test.jpg')
    print('Hoặc thay INPUT_IMAGE bằng đường dẫn ảnh khác.')
else:
    print(f'✅ Ảnh đầu vào: {INPUT_IMAGE}')

## 3. Phát hiện landmarks từ ảnh tĩnh

MediaPipe Face Mesh phát hiện **468 landmarks** bao gồm:
- Đường viền khuôn mặt (face contour)
- Mắt, lông mày, mũi, miệng
- Toạ độ 3D (x, y, z) được chuẩn hóa về [0,1]

In [ ]:
if INPUT_IMAGE.exists():
    mp_face_mesh = mp.solutions.face_mesh
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles

    image_bgr = cv2.imread(str(INPUT_IMAGE))
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w = image_bgr.shape[:2]
    print(f'Kích thước ảnh: {w}x{h}')

    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
    ) as face_mesh:
        results = face_mesh.process(image_rgb)

    if results.multi_face_landmarks:
        print(f'✅ Phát hiện {len(results.multi_face_landmarks)} khuôn mặt')
        face_landmarks = results.multi_face_landmarks[0]
        print(f'   Số landmarks: {len(face_landmarks.landmark)}')
    else:
        print('❌ Không phát hiện được khuôn mặt. Kiểm tra ảnh đầu vào.')

## 4. Visualize Landmarks

In [ ]:
if INPUT_IMAGE.exists() and results.multi_face_landmarks:
    annotated = draw_landmarks_on_image(image_bgr, face_landmarks)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(image_rgb)
    axes[0].set_title('Ảnh gốc', fontsize=14)
    axes[0].axis('off')

    axes[1].imshow(annotated_rgb)
    axes[1].set_title('Landmarks (468 điểm)', fontsize=14)
    axes[1].axis('off')

    plt.suptitle('MediaPipe Face Mesh - DECA_DHMT', fontsize=16)
    plt.tight_layout()
    plt.savefig('outputs/landmarks2d/notebook_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Đã lưu hình so sánh: outputs/landmarks2d/notebook_comparison.png')

## 5. Xem cấu trúc dữ liệu Landmark

In [ ]:
if INPUT_IMAGE.exists() and results.multi_face_landmarks:
    face_data = landmarks_to_dict(face_landmarks, w, h)
    print(f'Tổng số landmarks: {face_data["num_landmarks"]}')
    print('\n5 landmarks đầu tiên:')
    for lm in face_data['landmarks'][:5]:
        print(f'  #{lm["index"]:3d}: x={lm["x"]:.4f}, y={lm["y"]:.4f}, z={lm["z"]:.4f}'
              f'  -> px=({lm["x_px"]}, {lm["y_px"]})')

## 6. Lưu JSON & Chạy full process_image()

In [ ]:
if INPUT_IMAGE.exists():
    success = process_image(
        input_path=INPUT_IMAGE,
        output_dir=OUTPUT_DIR,
        min_detection_confidence=0.5,
        max_num_faces=1,
    )
    if success:
        json_path = OUTPUT_DIR / f'{INPUT_IMAGE.stem}_landmarks.json'
        data = load_json(json_path)
        print(f'✅ JSON đã lưu: {json_path}')
        print(f'   Số khuôn mặt: {len(data["faces"])}')
        print(f'   Kích thước ảnh: {data["image_size"]}')

## 7. Vẽ landmarks 3D (trục z)

MediaPipe cung cấp thêm chiều z (độ sâu tương đối), cho phép visualize 3D.

In [ ]:
if INPUT_IMAGE.exists() and results.multi_face_landmarks:
    from src.render.visualize import plot_landmarks_3d
    face_data = landmarks_to_dict(face_landmarks, w, h)
    plot_landmarks_3d(
        landmarks=face_data['landmarks'],
        output_path=Path('outputs/landmarks2d/landmarks_3d.png'),
    )